In [1]:
import sys, glob, os, time, warnings, html, re, zipfile
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, glob.glob("/opt/spark/python/lib/py4j*.zip")[0])

import numpy as np
import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import precision_recall_curve, average_precision_score

warnings.filterwarnings("ignore")

In [2]:
spark = (SparkSession.builder
    .appName("Export Dashboard")
    .master("local[4]")
    .config("spark.driver.memory", "5g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.hadoop.dfs.replication", "1")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 13:35:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Konfigurasi

In [3]:
HDFS = "hdfs://namenode:9000"
SOURCE_PATH   = f"{HDFS}/data/amazon_reviews/final_reviews_parquet"   # Clean Layer (sama seperti notebook 03 dan 04)
FEATURES_PATH = f"{HDFS}/data/ml/helpfulness_features"                # tabel fitur dari notebook 03
MODEL_DIR     = f"{HDFS}/data/ml/models"                              # model tersimpan dari notebook 03
PROD_RANK     = f"{HDFS}/data/graph/prod_rank"                        # hasil notebook 04
USER_RANK     = f"{HDFS}/data/graph/user_rank"                        # hasil notebook 04

OUT = "dashboard_data"
os.makedirs(OUT, exist_ok=True)

TOP_BRANDS      = 50    # merek di luar top-N digabung menjadi "Other brands"
MIN_REV_PRODUCT = 50    # produk dengan ulasan < ini digabung menjadi "Other products" (naikkan ke 100 jika file terlalu besar)
MIN_Q_REVIEWS   = 1000  # kuartal dengan ulasan lebih sedikit disembunyikan (sama seperti notebook 03)
BEST_MODEL      = {"A": "GBT", "B": "GBT"}   # model yang tersimpan di MODEL_DIR (cell 31 dan 32 notebook 03)


def save(df, name):
    path = os.path.join(OUT, name)
    df.to_csv(path, index=False, compression="gzip" if name.endswith(".gz") else None)
    print(f"{name:26s} {len(df):>9,} baris  {os.path.getsize(path) / 1e6:6.2f} MB")

In [4]:
raw = spark.read.parquet(SOURCE_PATH)
cols = ["reviewerID", "asin", "overall", "verified", "vote", "reviewText", "summary",
        "review_datetime", "brand", "main_cat", "title"]

# dedup identik dengan notebook 03 (cell 11)
reviews = (raw.select(*cols)
              .dropDuplicates(["reviewerID", "asin", "review_datetime", "reviewText", "summary", "overall"]))

base = (reviews
    .withColumn("year", F.year(F.to_date("review_datetime")))
    .withColumn("verified", F.coalesce(F.col("verified"), F.lit(False)))
    .withColumn("overall", F.col("overall").cast("int"))
    .withColumn("vote", F.coalesce(F.col("vote"), F.lit(0)))
    .withColumn("helpful", (F.col("vote") > 0).cast("int"))
    .withColumn("text_len", F.length(F.coalesce(F.col("reviewText"), F.lit(""))))
    .withColumn("main_cat", F.coalesce(F.col("main_cat"), F.lit("Unknown")))
    .withColumn("brand", F.coalesce(F.col("brand"), F.lit("Unknown")))
    .select("asin", "title", "year", "verified", "overall", "vote", "helpful", "text_len", "main_cat", "brand")
    .persist(StorageLevel.MEMORY_AND_DISK))

n_total = base.count()
print(f"Ulasan setelah dedup: {n_total:,}  (notebook 03: 2.488.658)")

[Stage 3:=================================================>       (14 + 2) / 16]

Ulasan setelah dedup: 2,488,658  (notebook 03: 2.488.658)


In [5]:
# --- merek: top-N berdasarkan jumlah ulasan, sisanya "Other brands" ("Unknown" dipertahankan)
top_brands = [r["brand"] for r in (base.filter(F.col("brand") != "Unknown")
                                       .groupBy("brand").count()
                                       .orderBy(F.desc("count")).limit(TOP_BRANDS).collect())]
brand_c = (F.when(F.col("brand").isin(top_brands + ["Unknown"]), F.col("brand"))
            .otherwise(F.lit("Other brands")))

# --- produk: hanya yang punya >= MIN_REV_PRODUCT ulasan dirinci per produk
elig = (base.groupBy("asin").count().filter(F.col("count") >= MIN_REV_PRODUCT)
            .select("asin").withColumn("is_elig", F.lit(True)))
n_elig = elig.count()
print(f"Produk yang dirinci (>= {MIN_REV_PRODUCT} ulasan): {n_elig:,}")

label = F.concat(F.coalesce(F.substring(F.col("title"), 1, 70), F.lit("(tanpa judul)")),
                 F.lit(" ("), F.col("asin"), F.lit(")"))
product_c = F.when(F.col("is_elig"), label).otherwise(F.lit("Other products"))

cube_main = (base.join(F.broadcast(elig), "asin", "left")
    .withColumn("brand", brand_c)
    .withColumn("product", product_c)
    .groupBy("year", "brand", "product", "main_cat", "verified", "overall")
    .agg(F.count("*").alias("n_reviews"),
         F.sum("helpful").alias("n_helpful"),
         F.sum("vote").alias("sum_vote"),
         F.sum("text_len").alias("sum_text_len")))

pm = cube_main.toPandas()
n_null = int(pm["year"].isna().sum())
if n_null:
    print(f"PERINGATAN: {n_null} baris cube tanpa tahun dibuang (tanggal tidak terbaca)")
    pm = pm.dropna(subset=["year"])
pm["year"] = pm["year"].astype(int)
for c in ("product", "brand"):                       # entity HTML seperti &eacute; dirapikan
    pm[c] = pm[c].map({v: html.unescape(v) for v in pm[c].unique()})
pm = pm.sort_values(["year", "brand", "product", "verified", "overall"])
save(pm, "cube_main.csv.gz")

Produk yang dirinci (>= 50 ulasan): 9,198


cube_main.csv.gz             304,257 baris    3.79 MB


In [6]:
len_group = (F.when(F.col("text_len") == 0, "0")
              .when(F.col("text_len") <= 100, "1-100")
              .when(F.col("text_len") <= 250, "101-250")
              .when(F.col("text_len") <= 500, "251-500")
              .when(F.col("text_len") <= 1000, "501-1000")
              .otherwise("1000+"))

cube_text = (base
    .withColumn("brand", brand_c)
    .withColumn("len_group", len_group)
    .groupBy("year", "brand", "main_cat", "verified", "overall", "len_group")
    .agg(F.count("*").alias("n_reviews"),
         F.sum("helpful").alias("n_helpful"),
         F.sum("text_len").alias("sum_text_len")))

ct = cube_text.toPandas().dropna(subset=["year"])
ct["year"] = ct["year"].astype(int)
ct["brand"] = ct["brand"].map({v: html.unescape(v) for v in ct["brand"].unique()})
ct = ct.sort_values(["year", "brand", "verified", "overall", "len_group"])
save(ct, "cube_text.csv.gz")

cube_text.csv.gz              48,248 baris    0.39 MB


In [7]:
# --- Pemeriksaan: total harus sama antar cube dan dengan jumlah baris
tot, tot_t = int(pm["n_reviews"].sum()), int(ct["n_reviews"].sum())
print(f"cube_main={tot:,} | cube_text={tot_t:,} | data={n_total:,} ->",
      "OK" if tot == tot_t == n_total else "BEDA! cek tahun kosong atau dedup")
assert tot == tot_t, "cube_main dan cube_text berbeda total"
assert int(pm["n_helpful"].sum()) == int(ct["n_helpful"].sum())

kpi = {
    "total_ulasan": tot,
    "total_vote": int(pm["sum_vote"].sum()),
    "rata_rating": round((pm["overall"] * pm["n_reviews"]).sum() / tot, 3),
    "persen_verified": round(pm.loc[pm["verified"], "n_reviews"].sum() / tot * 100, 1),
    "persen_helpful": round(pm["n_helpful"].sum() / tot * 100, 1),
    "persen_rating_5": round(pm.loc[pm["overall"] == 5, "n_reviews"].sum() / tot * 100, 1),
    "persen_ulasan_pada_produk_terinci": round(pm.loc[pm["product"] != "Other products", "n_reviews"].sum() / tot * 100, 1),
}
print(pd.Series(kpi).to_string())
print("Bandingkan dengan laporan: rating 5 = 58%, verified = 75,9%, rata-rata verified 4,10 vs unverified 3,77")

cube_main=2,488,658 | cube_text=2,488,658 | data=2,488,658 -> OK
total_ulasan                         2488658.000
total_vote                           3497926.000
rata_rating                                4.025
persen_verified                           76.400
persen_helpful                            17.000
persen_rating_5                           58.100
persen_ulasan_pada_produk_terinci         79.000
Bandingkan dengan laporan: rating 5 = 58%, verified = 75,9%, rata-rata verified 4,10 vs unverified 3,77


In [8]:
data = spark.read.parquet(FEATURES_PATH)
val  = data.filter(F.col("split") == "val")
test = data.filter(F.col("split") == "test")

# definisi skenario: salin dari notebook 03 (cell 21)
STATIK   = ["overall", "verified", "text_len", "word_count", "has_price"]
TEKS     = ["summary_len", "n_exclaim", "n_question", "n_sentences", "has_url",
            "has_linebreak", "upper_ratio", "digit_ratio", "avg_word_len"]
RIWAYAT  = ["user_prev_n", "user_prev_help"]
DEVIASI  = ["prod_prev_avg", "rating_dev_past", "abs_rating_dev_past"]
EKSPOSUR = ["tahun", "prod_prev_n", "prod_gap_days"]
SCENARIOS = {"A": STATIK + TEKS + RIWAYAT + DEVIASI + EKSPOSUR,
             "B": STATIK + TEKS + RIWAYAT + DEVIASI}

MODELS = {s: PipelineModel.load(f"{MODEL_DIR}/{s}_{BEST_MODEL[s]}") for s in SCENARIOS}


def collect_probabilities(model, df):
    return model.transform(df).select("label", vector_to_array("probability")[1].alias("p")).toPandas()


def best_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.argmax(f1))])


pred_val  = {s: collect_probabilities(MODELS[s], val)  for s in SCENARIOS}
pred_test = {s: collect_probabilities(MODELS[s], test) for s in SCENARIOS}
THRESH = {s: best_threshold(pred_val[s]["label"].values, pred_val[s]["p"].values) for s in SCENARIOS}

for s in SCENARIOS:
    ap = average_precision_score(pred_test[s]["label"], pred_test[s]["p"])
    print(f"Skenario {s}: PR-AUC test = {ap:.3f}, threshold validasi = {THRESH[s]:.3f}")
print("Acuan notebook 03 -> A: 0.683 / 0.686 | B: 0.620 / 0.633")

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
[Stage 76:==============================================>           (4 + 1) / 5]

Skenario A: PR-AUC test = 0.683, threshold validasi = 0.686
Skenario B: PR-AUC test = 0.620, threshold validasi = 0.633
Acuan notebook 03 -> A: 0.683 / 0.686 | B: 0.620 / 0.633


In [9]:
# --- ml_threshold.csv: tabel TP/FP/FN/TN per threshold untuk slider presisi vs cakupan
grid = np.round(np.arange(0.0, 1.0001, 0.01), 2)
rows = []
for s in SCENARIOS:
    y = pred_test[s]["label"].values.astype(int)
    p = pred_test[s]["p"].values
    pos = y == 1
    P, N = int(pos.sum()), int((~pos).sum())
    for th in grid:
        hat = p >= th
        tp = int((hat & pos).sum())
        fp = int((hat & ~pos).sum())
        rows.append({"skenario": s, "model": BEST_MODEL[s], "threshold": float(th),
                     "TP": tp, "FP": fp, "FN": P - tp, "TN": N - fp})
thr = pd.DataFrame(rows)
save(thr, "ml_threshold.csv")

# pemeriksaan pada threshold terpilih: bandingkan dengan notebook 03 (A: P=0.578 R=0.700 | B: P=0.516 R=0.689)
for s in SCENARIOS:
    hat = pred_test[s]["p"].values >= THRESH[s]
    y = pred_test[s]["label"].values.astype(int)
    tp = int((hat & (y == 1)).sum())
    print(f"Skenario {s} @ {THRESH[s]:.3f}: precision={tp / hat.sum():.3f} recall={tp / (y == 1).sum():.3f}")

ml_threshold.csv                 202 baris    0.01 MB
Skenario A @ 0.686: precision=0.578 recall=0.700
Skenario B @ 0.633: precision=0.516 recall=0.689


In [10]:
# --- ml_importance.csv: semua fitur GBT, diurutkan
imp_rows = []
for s, feats in SCENARIOS.items():
    names = feats + ["price_imp", "brand_idx", "main_cat_idx"]
    fi = MODELS[s].stages[-1].featureImportances.toArray()
    assert len(names) == len(fi), f"jumlah fitur {s} tidak cocok: {len(names)} vs {len(fi)}"
    imp_rows += [{"skenario": s, "fitur": n, "importance": round(float(v), 4)} for n, v in zip(names, fi)]
imp = (pd.DataFrame(imp_rows).sort_values(["skenario", "importance"], ascending=[True, False]))
save(imp, "ml_importance.csv")
print(imp.groupby("skenario").head(3).to_string(index=False))

ml_importance.csv                 47 baris    0.00 MB
skenario               fitur  importance
       A            text_len      0.3373
       A       prod_gap_days      0.1627
       A         prod_prev_n      0.1073
       B            text_len      0.3628
       B abs_rating_dev_past      0.1358
       B       prod_prev_avg      0.1268


In [11]:
# --- ml_quarterly.csv: aktual vs prediksi per kuartal (sama seperti notebook 03 cell 31)
recent = (data.filter("split in ('test', 'infer') and tgl >= '2015-01-01'")
              .withColumn("kuartal", F.concat(F.year("tgl").cast("string"), F.lit("Q"), F.quarter("tgl").cast("string"))))

summary = recent.groupBy("kuartal").agg(F.count("*").alias("n"), F.avg("label").alias("aktual"))
for s in SCENARIOS:
    pred = (MODELS[s].transform(recent)
                     .select("kuartal", vector_to_array("probability")[1].alias("p"))
                     .groupBy("kuartal")
                     .agg(F.avg((F.col("p") >= THRESH[s]).cast("int")).alias(f"pred_{s}")))
    summary = summary.join(pred, "kuartal")

qdf = summary.orderBy("kuartal").toPandas()
qdf = qdf[qdf["n"] >= MIN_Q_REVIEWS].copy()
for c in ("aktual", "pred_A", "pred_B"):
    qdf[c] = (qdf[c] * 100).round(2)
save(qdf, "ml_quarterly.csv")
print(qdf.to_string(index=False))

ml_quarterly.csv                  15 baris    0.00 MB
kuartal      n  aktual  pred_A  pred_B
 2015Q1  21967    7.63    6.74    8.78
 2015Q2  14915    8.30    7.88    9.96
 2015Q3  15539    8.62    8.42   10.38
 2015Q4  16771   13.53   14.06   15.32
 2016Q1  21272    9.49    9.16   10.81
 2016Q2  15811   10.73   10.34   12.30
 2016Q3  15282   10.37    9.70   11.50
 2016Q4  15820   10.12    9.15   11.28
 2017Q1 105761    7.63    5.21    7.87
 2017Q2  65957    7.63    4.19    8.17
 2017Q3  58008    4.68    3.22    7.77
 2017Q4  51738    5.47    3.95    9.10
 2018Q1  60888    2.72    2.01    6.74
 2018Q2  44034    1.91    2.20    6.96
 2018Q3  20994    1.40    2.45    7.11


In [12]:
import subprocess
os.environ["HOME"] = "/tmp"
for p in glob.glob("/tmp/.local/lib/python*/site-packages"):
    sys.path.insert(0, p)
try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "networkx"])
    for p in glob.glob("/tmp/.local/lib/python*/site-packages"):
        sys.path.insert(0, p)
    import networkx as nx
print("networkx", nx.__version__)

# --- bangun ulang daftar edge (identik dengan notebook 04 cell 11 dan 13; tanpa dedup tambahan)
df = spark.read.parquet(SOURCE_PATH).select("reviewerID", "asin", "overall", "vote")
edges_raw = df.groupBy("reviewerID", "asin").agg(F.count("*").alias("n_reviews"))
u_ok = edges_raw.groupBy("reviewerID").count().filter("count >= 2").select("reviewerID")
e1 = edges_raw.join(u_ok, "reviewerID")
p_ok = e1.groupBy("asin").count().filter("count >= 5").select("asin")
edges_f = e1.join(p_ok, "asin")
edges = (edges_f.select(F.concat(F.lit("U_"), "reviewerID").alias("src"),
                        F.concat(F.lit("P_"), "asin").alias("dst"))
                .persist(StorageLevel.MEMORY_AND_DISK))

n_edges = edges.count()
n_users = edges.select("src").distinct().count()
n_prods = edges.select("dst").distinct().count()
density = n_edges / (n_users * n_prods)
print(f"reviewer={n_users:,} produk={n_prods:,} edge={n_edges:,} densitas={density:.6f}")
print("Acuan notebook 04: 378.040 / 28.271 / 1.265.895 / 0.000118")
save(pd.DataFrame([{"n_users": n_users, "n_products": n_prods, "n_edges": n_edges, "density": density}]),
     "graph_summary.csv")

networkx 3.4.2


reviewer=378,040 produk=28,271 edge=1,265,895 densitas=0.000118
Acuan notebook 04: 378.040 / 28.271 / 1.265.895 / 0.000118
graph_summary.csv                  1 baris    0.00 MB


In [13]:
prod_pd = spark.read.parquet(PROD_RANK).toPandas()    # id, title, brand, n_reviewers, votes, mean_rating, pagerank
user_pd = spark.read.parquet(USER_RANK).toPandas()    # id, n_products, votes, mean_rating, pagerank

# --- semua produk
gp = prod_pd.copy()
gp["asin"] = gp["id"].str.replace(r"^P_", "", regex=True)
gp["title"] = gp["title"].fillna("(tanpa judul)").map(html.unescape)
gp["brand"] = gp["brand"].fillna("Unknown").map(html.unescape)
gp["votes"] = gp["votes"].fillna(0).astype(int)
gp["pr_rank"] = gp["pagerank"].rank(ascending=False, method="first").astype(int)
gp = gp[["asin", "title", "brand", "n_reviewers", "votes", "mean_rating", "pagerank", "pr_rank"]].sort_values("pr_rank")
save(gp.round({"mean_rating": 3, "pagerank": 3}), "graph_products.csv.gz")

# --- 1.000 reviewer teratas, anonim
top_u = user_pd.nlargest(1000, "pagerank").reset_index(drop=True)
gr = pd.DataFrame({"reviewer": [f"Reviewer {i + 1}" for i in range(len(top_u))],
                   "n_products": top_u["n_products"].astype(int),
                   "votes": top_u["votes"].fillna(0).astype(int),
                   "mean_rating": top_u["mean_rating"].round(3),
                   "pagerank": top_u["pagerank"].round(3)})
save(gr, "graph_reviewers.csv")

# --- sebaran derajat (untuk grafik log-log)
deg = pd.concat([
    prod_pd["n_reviewers"].value_counts().rename_axis("degree").reset_index(name="n_nodes").assign(type="produk"),
    user_pd["n_products"].value_counts().rename_axis("degree").reset_index(name="n_nodes").assign(type="reviewer")])
save(deg.sort_values(["type", "degree"]), "graph_degree.csv")

graph_products.csv.gz         28,271 baris    1.06 MB
graph_reviewers.csv            1,000 baris    0.03 MB
graph_degree.csv                 823 baris    0.01 MB


In [14]:
# --- proyeksi produk-produk, komunitas Louvain, betweenness (identik dengan notebook 04 cell 46)
topN = list(prod_pd.nlargest(300, "n_reviewers")["id"])
e_top = edges.filter(F.col("dst").isin(topN))

# batasi super-reviewer (> 100 produk) supaya tidak menghubungkan semua produk
ok_u = e_top.groupBy("src").count().filter("count BETWEEN 2 AND 100").select("src")
e_top = e_top.join(ok_u, "src").cache()

d = e_top.groupBy("dst").count().toPandas()
dp = dict(zip(d["dst"], d["count"]))

a, b = e_top.alias("a"), e_top.alias("b")
pairs = (a.join(b, (F.col("a.src") == F.col("b.src")) & (F.col("a.dst") < F.col("b.dst")))
          .groupBy(F.col("a.dst").alias("p1"), F.col("b.dst").alias("p2"))
          .agg(F.count("*").alias("shared")).toPandas())
pairs["jaccard"] = pairs["shared"] / (pairs["p1"].map(dp) + pairs["p2"].map(dp) - pairs["shared"])

G = nx.from_pandas_edgelist(pairs[pairs["jaccard"] >= 0.05], "p1", "p2", edge_attr=["jaccard", "shared"])
comm = nx.community.louvain_communities(G, weight="jaccard", seed=42)
cid = {n: i for i, c in enumerate(comm) for n in c}
bt = nx.betweenness_centrality(G)
pos = nx.spring_layout(G, weight="jaccard", seed=42, k=0.4)
print(f"Jaringan: {G.number_of_nodes()} produk, {G.number_of_edges()} sisi, {len(comm)} komunitas (laporan: 48)")

Jaringan: 206 produk, 233 sisi, 47 komunitas (laporan: 48)


In [15]:
info = prod_pd.set_index("id")
rows = []
for n in G.nodes:
    r = info.loc[n]
    rows.append({"asin": n[2:],
                 "title": html.unescape(r["title"]) if pd.notna(r["title"]) else "(tanpa judul)",
                 "brand": html.unescape(r["brand"]) if pd.notna(r["brand"]) else "Unknown",
                 "n_reviewers": int(r["n_reviewers"]),
                 "votes": int(r["votes"]) if pd.notna(r["votes"]) else 0,
                 "mean_rating": round(float(r["mean_rating"]), 3),
                 "pagerank": round(float(r["pagerank"]), 3),
                 "community": cid[n],
                 "betweenness": round(float(bt[n]), 5),
                 "degree": int(G.degree(n)),
                 "x": round(float(pos[n][0]), 5), "y": round(float(pos[n][1]), 5)})
nodes = pd.DataFrame(rows)

crow = []
for c, g in nodes.groupby("community"):
    g = g.sort_values("pagerank", ascending=False)
    crow.append({"community": int(c),
                 "label": f"K{c} - {g['title'].iloc[0][:30]}",
                 "n_products": len(g),
                 "avg_rating": round(float(g["mean_rating"].mean()), 2),
                 "top_members": "; ".join(t[:30] for t in g["title"].head(4)),
                 "bridge_product": g.sort_values("betweenness", ascending=False)["title"].iloc[0][:40]})
comm_df = pd.DataFrame(crow).sort_values("n_products", ascending=False)
nodes["community_label"] = nodes["community"].map(dict(zip(comm_df["community"], comm_df["label"])))

edges_df = pd.DataFrame([(u[2:], v[2:], round(float(dd["jaccard"]), 4), int(dd["shared"]))
                         for u, v, dd in G.edges(data=True)], columns=["p1", "p2", "jaccard", "shared"])

save(nodes, "graph_nodes.csv")
save(edges_df, "graph_edges.csv")
save(comm_df, "graph_communities.csv")
print(comm_df[["label", "n_products", "bridge_product"]].head(8).to_string(index=False))

graph_nodes.csv                  206 baris    0.03 MB
graph_edges.csv                  233 baris    0.01 MB
graph_communities.csv             47 baris    0.01 MB
                               label  n_products                           bridge_product
K11 - Destiny - Standard Edition - P          25 Assassin's Creed Unity Limited Edition -
 K5 - Xbox One Wireless Controller (          13        Halo: The Master Chief Collection
K16 - Halo 4 - Xbox 360 (Standard Ga          13                              Halo - Xbox
     K39 - New Super Mario Bros. Wii          13                       Super Mario Galaxy
K17 - Assassin's Creed - Playstation          12         Assassin's Creed - Playstation 3
K31 - Super Smash Bros. - Nintendo 3          10        Pokémon Omega Ruby - Nintendo 3DS
 K12 - Mario Kart 8 - Nintendo Wii U          10            Mario Kart 8 - Nintendo Wii U
                 K1 - Kingdom Hearts           8 Grand Theft Auto: San Andreas - PlayStat


In [16]:
# tidak boleh ada kolom teks ulasan atau ID reviewer
forbidden = {"reviewerID", "reviewerName", "reviewText", "summary", "src", "id"}
for fn in sorted(os.listdir(OUT)):
    if fn.endswith((".csv", ".csv.gz")):
        cols_ = set(pd.read_csv(os.path.join(OUT, fn), nrows=1).columns)
        assert not (cols_ & forbidden), f"{fn} memuat kolom terlarang: {cols_ & forbidden}"
assert gr["reviewer"].str.fullmatch(r"Reviewer \d+").all()
print("Pemeriksaan privasi lolos.")

zip_path = "dashboard_data.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn), fn)
print(f"{zip_path}: {os.path.getsize(zip_path) / 1e6:.1f} MB")
print(sorted(os.listdir(OUT)))

Pemeriksaan privasi lolos.
dashboard_data.zip: 5.3 MB
['cube_main.csv.gz', 'cube_text.csv.gz', 'graph_communities.csv', 'graph_degree.csv', 'graph_edges.csv', 'graph_nodes.csv', 'graph_products.csv.gz', 'graph_reviewers.csv', 'graph_summary.csv', 'ml_importance.csv', 'ml_quarterly.csv', 'ml_threshold.csv']
